# NTK Permutation Learning Experiment

This notebook trains Neural Tangent Kernel (NTK) MLPs to learn permutation matrices.

## Overview
- **Experiment 1**: Find minimum N (ensemble size) to reach target accuracy
- **Experiment 2**: Track accuracy growth curve as ensemble size increases

## 1. Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import csv
from datetime import datetime
from pathlib import Path
from tqdm import tqdm

# Automatically select GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Dataset Classes and Functions

In [ ]:
class PermutationDataset(Dataset):
    def __init__(self, P: torch.Tensor):
        super().__init__()
        self.P = P
        self.n = P.shape[0]
        # Huấn luyện trên các vector cơ sở trực giao [cite: 280]
        self.basis = torch.eye(self.n)

    def __len__(self):
        return self.n

    def __getitem__(self, idx: int):
        x = self.basis[idx]
        y = self.P @ x
        return x, y

def generate_permutation(n):
    """Generate a random permutation matrix."""
    P = torch.eye(n)[torch.randperm(n)]
    return P

def generate_test_set(k, num_samples, nnz=2):
    """Generate test set with nnz bits set to 1 (default is 2 as in Exp 2) [cite: 488]."""
    x_test = torch.zeros((num_samples, k))
    for i in range(num_samples):
        indices = torch.randperm(k)[:nnz]
        x_test[i, indices] = 1.0
    
    # Normalize input by sqrt(nnz)
    x_test_norm = x_test / (nnz ** 0.5)
    return x_test_norm, x_test  # Return both normalized and raw bit versions

## 3. Model Definition

In [ ]:
class NTKMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, sigma_w=1.0):
        super(NTKMLP, self).__init__()
        # Paper uses 2-layer network without bias
        self.fc1 = nn.Linear(input_dim, hidden_dim, bias=False)
        self.fc2 = nn.Linear(hidden_dim, output_dim, bias=False)
        self.relu = nn.ReLU()
        self.sigma_w = sigma_w
        self._initialize_weights()

    def _initialize_weights(self):
        """Initialize weights according to NTK formula: std = sigma / sqrt(fan_in)"""
        nn.init.normal_(self.fc1.weight, mean=0, std=self.sigma_w / (self.fc1.in_features ** 0.5))
        nn.init.normal_(self.fc2.weight, mean=0, std=self.sigma_w / (self.fc2.in_features ** 0.5))

    def forward(self, x):
        """F(x) = W2 * ReLU(W1 * x) [cite: 117]"""
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

## 4. Training and Evaluation Functions

In [ ]:
def train_single_model(train_loader, k, hidden_dim, epochs=2500, lr=0.1, show_progress=False):
    """
    Train a single NTKMLP model on basis vectors.
    2-layer network architecture without bias.
    """
    model = NTKMLP(k, hidden_dim, k).to(device)
    optimizer = optim.SGD(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    
    model.train()
    epoch_iterator = tqdm(range(epochs), desc="    Training", leave=False, disable=not show_progress)
    for epoch in epoch_iterator:
        total_loss = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        if show_progress and epoch % 100 == 0:
            epoch_iterator.set_postfix({'loss': f'{total_loss:.6f}'})
    
    return model.eval(), total_loss

def get_accuracy(all_preds, Y_truth):
    """
    Calculate accuracy using Ensemble Mean G(x) > 0.
    """
    ensemble_mean = torch.stack(all_preds).mean(dim=0)
    predictions = (ensemble_mean > 0).float()
    return (predictions == Y_truth.to(device)).all(dim=1).float().mean().item()

## 5. Main Experiment Function

In [ ]:
def run_combined_experiment(k_list, target_acc, trials, epochs, output_dir, hidden_dim_override=None, lr=0.1):
    """
    Run combined experiments:
    1. Experiment 1: Find minimum N to reach target_acc.
    2. Experiment 2: Save accuracy growth data for plotting curves.
    
    Args:
        hidden_dim_override: If provided, use this value instead of k*1000 for ALL K values
    """
    print(f"\n{'='*60}")
    print(f"Combined Experiment: Required N & Accuracy Curve")
    print(f"Target Accuracy: {target_acc}")
    print(f"Trials per K: {trials}")
    print(f"{'='*60}\n")
    
    for k in k_list:
        print(f"\nProcessing K = {k}")
        # Allow override hidden_dim, otherwise use k * 1000
        hidden_dim = hidden_dim_override if hidden_dim_override is not None else k * 1000
        print(f"Hidden dim: {hidden_dim}")
        
        # Create folder for this K
        k_dir = output_dir / f"k_{k}"
        k_dir.mkdir(parents=True, exist_ok=True)
        
        # 1. File summary.csv: Only save stopping point (Required N) for each trial
        summary_path = k_dir / "summary.csv"
        with open(summary_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['trial_id', 'required_n', 'final_accuracy'])
        
        # 2. File accuracy_progress.csv: Save ENTIRE process (for Experiment 2)
        progress_path = k_dir / "accuracy_progress.csv"
        with open(progress_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['trial_id', 'n', 'accuracy'])
        
        n_per_trial = []
        last_permutation = None
        
        for t in range(trials):
            P = generate_permutation(k)
            last_permutation = P
            
            train_loader = DataLoader(
                PermutationDataset(P), 
                batch_size=k
            )
            
            # Generate test set with nnz=2
            X_test_norm, X_test_raw = generate_test_set(k, num_samples=200, nnz=2)
            Y_test = (P @ X_test_raw.T).T.to(device)
            X_test_norm = X_test_norm.to(device)
            
            all_preds = []
            acc = 0
            n = 0
            
            # Progress bar for each Trial
            pbar = tqdm(total=None, desc=f"  K={k} Trial {t+1}/{trials}", unit="model")
            
            # Open progress file to append data continuously (in case of crash)
            with open(progress_path, 'a', newline='', encoding='utf-8') as f_prog:
                prog_writer = csv.writer(f_prog)
                
                while acc < target_acc and n < 5000:
                    n += 1
                    # Show progress every 10th model to monitor training
                    show_progress = (n % 10 == 1)
                    model, final_loss = train_single_model(train_loader, k, hidden_dim, epochs, lr=lr, show_progress=show_progress)
                    
                    # Warn if model didn't converge properly
                    if final_loss > 0.01:
                        print(f"\n    ⚠️  WARNING: Model {n} has high final loss={final_loss:.6f} - may not have converged!")
                    
                    with torch.no_grad():
                        pred = model(X_test_norm).detach()
                        all_preds.append(pred)
                    
                    acc = get_accuracy(all_preds, Y_test)
                    
                    # SAVE EXPERIMENT 2 DATA: Save accuracy at each step n
                    prog_writer.writerow([t + 1, n, f'{acc:.6f}'])
                    
                    pbar.update(1)
                    pbar.set_postfix({'acc': f'{acc:.4f}'})
                    
                    if acc >= target_acc:
                        break
            
            pbar.close()
            n_per_trial.append(n)
            
            # SAVE EXPERIMENT 1 DATA: Save Required N for this trial
            with open(summary_path, 'a', newline='', encoding='utf-8') as f_sum:
                sum_writer = csv.writer(f_sum)
                sum_writer.writerow([t + 1, n, f'{acc:.6f}'])
            
            print(f"    -> Trial {t+1} completed at N = {n}")

        # Save final statistics for K
        avg_n = sum(n_per_trial) / trials
        with open(k_dir / "final_results.txt", 'w') as f:
            f.write(f"Avg Required N: {avg_n:.2f}\n")
            f.write(f"Trials: {trials}\n")
            f.write(f"Epochs: {epochs}\n")
        
        if last_permutation is not None:
            torch.save(last_permutation, k_dir / "permutation.pt")

## 6. Configuration and Run Experiment

Configure your experiment parameters below and run the cell to start training.

In [ ]:
# ============ EXPERIMENT CONFIGURATION ============

# K values to test (input dimensions)
k_list = [5, 10, 15, 20]

# Number of trials per K value
trials = 5

# Number of epochs per model training
epochs = 1250

# Target accuracy to reach (default: 0.9)
target_acc = 0.9

# Fixed hidden dimension for all K values (None = auto-scale to k*1000)
hidden_dim = None  # or set to a specific value like 5000

# Learning rate for SGD optimizer
lr = 0.1

# ============ RUN EXPERIMENT ============

# Create output directory with timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = Path("results") / f"combined_run_{timestamp}"
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Data will be saved to: {output_dir}")
if hidden_dim:
    print(f"Using fixed hidden_dim = {hidden_dim} for all K values")
else:
    print(f"Using auto-scaling hidden_dim = k * 1000")
print(f"Learning rate: {lr}")

# Run the experiment
run_combined_experiment(k_list, target_acc, trials, epochs, output_dir, hidden_dim, lr)

print("\n" + "="*60)
print("EXPERIMENT COMPLETED!")
print(f"Results saved to: {output_dir}")
print("="*60)